In [3]:
import pandas as pd

calendar_path = "C:/Users/Elize/Documents/summer/DATA VISUALIZATION/calendar.csv"
listings_path = "C:/Users/Elize/Documents/summer/DATA VISUALIZATION/listings.csv/listings.csv"
reviews_path = "C:/Users/Elize/Documents/summer/DATA VISUALIZATION/reviews.csv/reviews.csv"

calendar_df = pd.read_csv(calendar_path)
listings_df = pd.read_csv(listings_path)
reviews_df = pd.read_csv(reviews_path)

calendar_df['date'] = pd.to_datetime(calendar_df['date'], errors='coerce')
calendar_df['price'] = calendar_df['price'].replace('[\$,]', '', regex=True).astype(float)
calendar_filtered = calendar_df[(calendar_df['date'] >= '2025-03-01') & (calendar_df['available'] == True)]

avg_price_per_listing = (
    calendar_filtered
    .groupby('listing_id')['price']
    .mean()
    .reset_index()
    .rename(columns={'price': 'avg_price_per_listing_2025'})
)

reviews_df['date'] = pd.to_datetime(reviews_df['date'], errors='coerce')
reviews_recent = reviews_df[reviews_df['date'] >= '2025-03-01']
reviews_recent_unique = reviews_recent.drop_duplicates(subset=['listing_id', 'date'])

recent_review_count = (
    reviews_recent_unique
    .groupby('listing_id')
    .size()
    .reset_index(name='recent_review_count')
)

listings_df['price'] = listings_df['price'].replace('[\$,]', '', regex=True).astype(float)

listings_filtered = listings_df[[
    'id', 'neighbourhood_cleansed', 'price', 'availability_365',
    'number_of_reviews', 'review_scores_rating'
]].rename(columns={'id': 'listing_id'})

merged_df = listings_filtered.merge(avg_price_per_listing, on='listing_id', how='left')
merged_df = merged_df.merge(recent_review_count, on='listing_id', how='left')

neighbourhood_summary = (
    merged_df
    .groupby('neighbourhood_cleansed')
    .agg(
        active_listings=('listing_id', 'count'),
        avg_nightly_price=('price', 'mean'),
        avg_review_score=('review_scores_rating', 'mean'),
        total_recent_reviews=('recent_review_count', 'sum')
    )
    .reset_index()
)

print(neighbourhood_summary)



<>:12: SyntaxWarning: invalid escape sequence '\$'
<>:34: SyntaxWarning: invalid escape sequence '\$'
<>:12: SyntaxWarning: invalid escape sequence '\$'
<>:34: SyntaxWarning: invalid escape sequence '\$'
C:\Users\Elize\AppData\Local\Temp\ipykernel_39816\2913202764.py:12: SyntaxWarning: invalid escape sequence '\$'
  calendar_df['price'] = calendar_df['price'].replace('[\$,]', '', regex=True).astype(float)
C:\Users\Elize\AppData\Local\Temp\ipykernel_39816\2913202764.py:34: SyntaxWarning: invalid escape sequence '\$'
  listings_df['price'] = listings_df['price'].replace('[\$,]', '', regex=True).astype(float)
C:\Users\Elize\AppData\Local\Temp\ipykernel_39816\2913202764.py:7: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  calendar_df = pd.read_csv(calendar_path)


    neighbourhood_cleansed  active_listings  avg_nightly_price  \
0                    Acton               14         125.000000   
1          Adams-Normandie               66          75.338983   
2             Agoura Hills               50         300.295455   
3               Agua Dulce               19         392.277778   
4                 Alhambra              705         154.370618   
..                     ...              ...                ...   
261            Willowbrook               44         150.350000   
262             Wilmington               17         123.750000   
263         Windsor Square               26         153.300000   
264               Winnetka              126         173.463636   
265         Woodland Hills              535         390.682609   

     avg_review_score  total_recent_reviews  
0            4.928182                   0.0  
1            4.649655                   0.0  
2            4.844146                   0.0  
3            4.766875  